# LogBasedProdInsigh：基于设备 Log 的生产状态数据处理

本 Notebook 用于将原始设备 Log 数据处理成后续可用于生产状态回放、参数观察和异常分析的数据底座。

当前阶段只做数据处理，不做图表、不做 AI、不做异常检测。

处理目标：

1. 读取 data 目录下的原始设备 Log；
2. 标准化时间字段；
3. 处理重复列名；
4. 保留标准化宽表；
5. 将 L1 / L2 两条产线转换为带 line 字段的长表；
6. 将设备共有状态参数加 device_ 前缀后合并到长表；
7. 输出字段映射表和字段质量画像表。

In [ ]:
# =========================
# 配置区
# =========================
RAW_DATA_FILENAME = "M04.xlsx"
RAW_SHEET_NAME = None

OUTPUT_DIR_NAME = "已处理"

OUTPUT_WIDE_FILENAME = "processed_equipment_log_standardized_wide.xlsx"
OUTPUT_LONG_FILENAME = "processed_equipment_log_line_state_long.xlsx"
OUTPUT_DEVICE_STATE_FILENAME = "processed_equipment_log_device_state.xlsx"
OUTPUT_FIELD_MAPPING_FILENAME = "processed_equipment_log_field_mapping.xlsx"
OUTPUT_FIELD_PROFILE_FILENAME = "processed_equipment_log_field_profile.xlsx"

CREATE_ONOFF_NUMERIC_COLUMNS = True

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = DATA_DIR / OUTPUT_DIR_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE_PATH = DATA_DIR / RAW_DATA_FILENAME
WIDE_OUTPUT_PATH = OUTPUT_DIR / OUTPUT_WIDE_FILENAME
LONG_OUTPUT_PATH = OUTPUT_DIR / OUTPUT_LONG_FILENAME
DEVICE_STATE_OUTPUT_PATH = OUTPUT_DIR / OUTPUT_DEVICE_STATE_FILENAME
FIELD_MAPPING_OUTPUT_PATH = OUTPUT_DIR / OUTPUT_FIELD_MAPPING_FILENAME
FIELD_PROFILE_OUTPUT_PATH = OUTPUT_DIR / OUTPUT_FIELD_PROFILE_FILENAME

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw file path: {RAW_FILE_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
def make_unique_columns(cols):
    counters = {}
    unique_cols = []
    raw_to_unique = []
    duplicate_summary = []
    for col in cols:
        col = str(col)
        if col not in counters:
            counters[col] = 0
            unique_col = col
        else:
            counters[col] += 1
            unique_col = f"{col}_{counters[col]}"
        unique_cols.append(unique_col)
        raw_to_unique.append((col, unique_col))
    for col, count in counters.items():
        if count > 0:
            duplicate_summary.append({"source_col": col, "duplicate_count": count + 1})
    return unique_cols, raw_to_unique, pd.DataFrame(duplicate_summary)

def strip_line_prefix(col_name):
    return re.sub(r"^L[12][\s_-]+", "", col_name).strip()

def make_unique_names(names):
    counters = {}
    out = []
    for n in names:
        if n not in counters:
            counters[n] = 0
            out.append(n)
        else:
            counters[n] += 1
            out.append(f"{n}_{counters[n]}")
    return out

def safe_device_name(base_name, occupied):
    candidate = f"device_{base_name}"
    i = 0
    while candidate in occupied:
        i += 1
        candidate = f"device_{base_name}_{i}"
    occupied.add(candidate)
    return candidate

In [ ]:
if RAW_SHEET_NAME is None:
    df_raw = pd.read_excel(RAW_FILE_PATH)
    sheet_info = "first sheet (default)"
else:
    df_raw = pd.read_excel(RAW_FILE_PATH, sheet_name=RAW_SHEET_NAME)
    sheet_info = RAW_SHEET_NAME
print(f"原始文件路径: {RAW_FILE_PATH}")
print(f"sheet 名: {sheet_info}")
print(f"原始数据行数: {len(df_raw)}")
print(f"原始数据列数: {df_raw.shape[1]}")

In [ ]:
original_cols = [str(c) for c in df_raw.columns]
unique_cols, raw_to_unique, duplicate_summary_df = make_unique_columns(original_cols)
df = df_raw.copy()
df.columns = unique_cols
print(f"重复列（按原始列名统计）数量: {len(duplicate_summary_df)}")
if len(duplicate_summary_df) > 0:
    display(duplicate_summary_df)
raw_unique_df = pd.DataFrame(raw_to_unique, columns=["source_col", "source_col_unique"])

In [ ]:
lower_map = {c.lower(): c for c in df.columns}
date_col = lower_map.get("date")
time_col = lower_map.get("time")
if date_col is not None and time_col is not None:
    timestamp_series = pd.to_datetime(df[date_col].astype(str).str.strip() + " " + df[time_col].astype(str).str.strip(), errors="coerce")
else:
    ts_candidates = [c for c in df.columns if c.lower() in ["timestamp", "datetime", "date_time"]]
    if not ts_candidates:
        raise ValueError("未找到可用时间字段：请至少包含 date + Time 或 timestamp 列。")
    timestamp_series = pd.to_datetime(df[ts_candidates[0]], errors="coerce")
if "timestamp" in df.columns:
    df = df.drop(columns=["timestamp"])
df.insert(0, "timestamp", timestamp_series)
df = df.sort_values("timestamp", ascending=True).reset_index(drop=True)
invalid_ts_count = df["timestamp"].isna().sum()
print(f"时间范围: {df['timestamp'].min()} ~ {df['timestamp'].max()}")
print(f"无法解析 timestamp 的记录数: {invalid_ts_count}")
print("采样记录:")
display(df[["timestamp"] + ([date_col] if date_col else []) + ([time_col] if time_col else [])].head())

In [ ]:
time_cols = [c for c in ["timestamp", date_col, time_col] if c is not None and c in df.columns]
l1_pattern = re.compile(r"^L1[\s_-]+")
l2_pattern = re.compile(r"^L2[\s_-]+")
l1_cols = [c for c in df.columns if l1_pattern.search(c)]
l2_cols = [c for c in df.columns if l2_pattern.search(c)]
device_state_cols = [c for c in df.columns if c not in set(time_cols + l1_cols + l2_cols)]
print(f"L1 字段数量: {len(l1_cols)}")
print(f"L2 字段数量: {len(l2_cols)}")
print(f"设备共有状态字段数量: {len(device_state_cols)}")

In [ ]:
wide_df = df.copy()
wide_df.to_excel(WIDE_OUTPUT_PATH, index=False)
print(f"已输出标准化宽表: {WIDE_OUTPUT_PATH}")

In [ ]:
device_state_df = df[["timestamp"] + device_state_cols].copy()
occupied_names = set()
device_name_map = {}
for col in device_state_cols:
    mapped = safe_device_name(col, occupied_names)
    device_name_map[col] = mapped
if device_state_cols:
    device_state_df = device_state_df.rename(columns=device_name_map)
device_state_df.to_excel(DEVICE_STATE_OUTPUT_PATH, index=False)
print(f"已输出设备共有状态表: {DEVICE_STATE_OUTPUT_PATH}")

In [ ]:
def build_line_long(df_input, line_cols, line_name):
    part = df_input[["timestamp"] + ([date_col] if date_col else []) + ([time_col] if time_col else []) + line_cols].copy()
    base_names = [strip_line_prefix(c) for c in line_cols]
    base_names = make_unique_names(base_names)
    rename_map = {old: new for old, new in zip(line_cols, base_names)}
    part = part.rename(columns=rename_map)
    part.insert(1, "line", line_name)
    if CREATE_ONOFF_NUMERIC_COLUMNS:
        for c in list(part.columns):
            if c.endswith("OnOff") or " OnOff" in c or "_OnOff" in c:
                part[f"{c}__num"] = part[c].map({"On": 1, "Off": 0})
    return part, rename_map
l1_long, l1_rename = build_line_long(df, l1_cols, "L1")
l2_long, l2_rename = build_line_long(df, l2_cols, "L2")
line_state_long = pd.concat([l1_long, l2_long], ignore_index=True)
line_state_long = line_state_long.merge(device_state_df, on="timestamp", how="left")
line_state_long = line_state_long.sort_values(["timestamp", "line"], ascending=True).reset_index(drop=True)
line_state_long.to_excel(LONG_OUTPUT_PATH, index=False)
print(f"已输出产线状态长表: {LONG_OUTPUT_PATH}")
print(f"长表行数: {len(line_state_long)} (原始行数: {len(df)})")

In [ ]:
mapping_rows = []
for src, src_unique in raw_to_unique:
    mapping_rows.append({"source_col": src, "source_col_unique": src_unique, "target_col": src_unique, "group": "wide", "line": "", "base_name": "", "target_table": "standardized_wide", "note": "标准化宽表保留字段"})
for c in l1_cols:
    mapping_rows.append({"source_col": c, "source_col_unique": c, "target_col": l1_rename[c], "group": "line_state", "line": "L1", "base_name": l1_rename[c], "target_table": "line_state_long", "note": "L1 产线参数"})
for c in l2_cols:
    mapping_rows.append({"source_col": c, "source_col_unique": c, "target_col": l2_rename[c], "group": "line_state", "line": "L2", "base_name": l2_rename[c], "target_table": "line_state_long", "note": "L2 产线参数"})
for src, tgt in device_name_map.items():
    mapping_rows.append({"source_col": src, "source_col_unique": src, "target_col": tgt, "group": "device_state", "line": "device", "base_name": src, "target_table": "device_state / line_state_long", "note": "设备共有状态参数"})
field_mapping_df = pd.DataFrame(mapping_rows)
field_mapping_df.to_excel(FIELD_MAPPING_OUTPUT_PATH, index=False)
print(f"已输出字段映射表: {FIELD_MAPPING_OUTPUT_PATH}")

In [ ]:
def profile_dataframe(df_input, group_name, line_name="", base_name_map=None):
    rows = []
    for c in df_input.columns:
        s = df_input[c]
        s_non_na = s.dropna()
        is_numeric = pd.api.types.is_numeric_dtype(s)
        is_onoff = s_non_na.astype(str).isin(["On", "Off"]).all() if len(s_non_na) else False
        rows.append({
            "field_name": c,
            "group": group_name,
            "line": line_name,
            "base_name": base_name_map.get(c, c) if base_name_map else c,
            "dtype": str(s.dtype),
            "missing_count": int(s.isna().sum()),
            "missing_ratio": float(s.isna().mean()),
            "unique_count": int(s.nunique(dropna=True)),
            "is_all_missing": bool(s.isna().all()),
            "is_numeric": bool(is_numeric),
            "is_onoff": bool(is_onoff),
            "min": s.min() if is_numeric else np.nan,
            "max": s.max() if is_numeric else np.nan,
            "mean": s.mean() if is_numeric else np.nan,
            "std": s.std() if is_numeric else np.nan,
        })
    return rows
profile_rows = []
profile_rows += profile_dataframe(wide_df, "wide")
profile_rows += profile_dataframe(line_state_long, "line_state_long")
profile_rows += profile_dataframe(device_state_df, "device_state")
field_profile_df = pd.DataFrame(profile_rows)
field_profile_df.to_excel(FIELD_PROFILE_OUTPUT_PATH, index=False)
print(f"已输出字段质量画像表: {FIELD_PROFILE_OUTPUT_PATH}")

In [ ]:
print("标准化宽表路径:", WIDE_OUTPUT_PATH)
print("产线状态长表路径:", LONG_OUTPUT_PATH)
print("设备共有状态表路径:", DEVICE_STATE_OUTPUT_PATH)
print("字段映射表路径:", FIELD_MAPPING_OUTPUT_PATH)
print("字段质量画像表路径:", FIELD_PROFILE_OUTPUT_PATH)
print("\n标准化宽表维度:", wide_df.shape)
print("产线状态长表维度:", line_state_long.shape)
print("设备共有状态表维度:", device_state_df.shape)
display(wide_df.head())
display(line_state_long.head())
display(device_state_df.head())
display(field_mapping_df.head())
display(field_profile_df.head())